# 02 · Popularity plus a residual

Popularity is hard to beat: it knows each card's track record directly, and an embedding model has to rebuild that
from the text. Here popularity is an input instead: `score = a·log(popularity rate) + model(black, white)`, so the
model only has to learn what popularity misses.

The training examples use leave-one-out rates (a pick isn't counted in its own card's rate), otherwise the model
learns to trust popularity more than it deserves on unseen games.

Sanity check: *popularity, learned scale* must match the popularity baseline's MRR (scaling doesn't change the order);
only its log loss can differ.

In [ ]:
import os
import sys
from functools import partial
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "ml").is_dir() else Path.cwd().parent  # papermill: repo root; VS Code: notebooks/
sys.path.insert(0, str(ROOT))
os.environ.setdefault("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")

import torch
from torch import nn

from ml import experiments as ex
from ml import lower_priority
from ml.train import TrainConfig

print("CPU priority:", lower_priority())  # idle CPU only: the game comes first

In [ ]:
snapshot = None
embedding = "e5"
variants = [            # (name, rank of the bilinear term, white-card term?)
    ("popularity, learned scale", 0, False),
    ("popularity + white", 0, True),
    ("popularity + bilinear r4", 4, True),
    ("popularity + bilinear r16", 16, True),
]
center = True
lr = 1e-2
weight_decay = 1e-3
max_epochs = 600
patience = 25
log_mlflow = True

In [ ]:
data = ex.load(snapshot)
print(data.snapshot.name, data.sizes())
inputs, targets = data.tensors(embedding)
if center:
    inputs = ex.center(inputs)
dim = inputs["train"][0].shape[-1]
train_ex = data.splits["train"]
log_pop = {"train": ex.popularity_log_rates(train_ex, train_ex, leave_one_out=True),
           "val": ex.popularity_log_rates(train_ex, data.splits["val"]),
           "test": ex.popularity_log_rates(train_ex, data.splits["test"])}
inputs = {s: (inputs[s][0], torch.tensor(log_pop[s], dtype=torch.float32)) for s in inputs}

In [ ]:
class PopularityPlus(nn.Module):
    """score = a·log(popularity rate) [+ w·white] [+ (P·black)·(Q·white)]."""

    def __init__(self, dim, rank, white):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(()))
        self.white = nn.Linear(dim, 1) if white else None
        self.p = nn.Linear(dim, rank, bias=False) if rank else None
        self.q = nn.Linear(dim, rank, bias=False) if rank else None

    def forward(self, x, log_pop):
        black, hand = x[:, :1], x[:, 1:]
        score = self.scale * log_pop
        if self.white is not None:
            score = score + self.white(hand).squeeze(-1)
        if self.p is not None:
            score = score + (self.p(black) * self.q(hand)).sum(-1)
        return score

In [ ]:
config = TrainConfig(lr=lr, weight_decay=weight_decay, max_epochs=max_epochs, patience=patience)
results = ex.baselines(data)
for name, rank, white in variants:
    results.append(ex.fit(f"{name} {embedding}", partial(PopularityPlus, dim, rank, white), inputs, targets, config,
                          {"model_type": "popularity-residual", "rank": rank, "white_term": white,
                           "embedding": embedding, "center": center}))
ex.table(results)

In [ ]:
if log_mlflow:
    print("MLflow parent run:", ex.log("02_popularity_residual", data, results, {"embedding": embedding, "center": center}))